# 🐛 Python Code Bug Detection System
**Built from scratch | Google Colab | CodeBERT + LSTM**

## What this notebook does
1. Generates a synthetic dataset of clean vs buggy Python code
2. Preprocesses and tokenizes the code
3. Trains two models: a BiLSTM baseline and a CodeBERT fine-tuned model
4. Evaluates both with accuracy, F1, confusion matrix
5. Launches a Gradio demo where you paste code and get a bug score

## Before you start
**IMPORTANT: Enable GPU**
- Go to `Runtime` → `Change runtime type` → select **T4 GPU** → Save
- This is free on Google Colab and speeds up training from hours to minutes

---

## Cell 1 — Install dependencies
Run this first. It installs everything needed. Takes ~2 minutes.

In [ ]:
# Install all required packages
!pip install transformers datasets gradio torch scikit-learn matplotlib seaborn --quiet

print("✅ All packages installed successfully!")

## Cell 2 — Imports and GPU check

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import random
import ast
import tokenize
import io
import re
import json
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

---
## Phase 1 — Dataset Generation

We create a synthetic dataset with two sources:
- **Clean code**: correct Python functions
- **Buggy code**: the same functions with common bugs injected

Common bugs we inject:
- Off-by-one errors (`range(n)` → `range(n+1)`)
- Wrong operator (`==` → `!=`, `>` → `<`)
- Wrong return value
- Missing `return` statement
- Wrong variable name
- Index errors (hardcoded wrong index)

> **Optional real dataset**: After this notebook works, you can swap the synthetic data
> for the CodeSearchNet dataset. Instructions are at the bottom of this cell.

In [ ]:
# ─────────────────────────────────────────────
# Clean Python code examples (label = 0)
# ─────────────────────────────────────────────
CLEAN_EXAMPLES = [
    # Sorting / searching
    """def find_max(lst):
    if not lst:
        return None
    max_val = lst[0]
    for item in lst[1:]:
        if item > max_val:
            max_val = item
    return max_val""",

    """def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1""",

    """def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr""",

    # Math functions
    """def factorial(n):
    if n == 0 or n == 1:
        return 1
    return n * factorial(n - 1)""",

    """def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True""",

    """def gcd(a, b):
    while b:
        a, b = b, a % b
    return a""",

    """def fibonacci(n):
    if n <= 0:
        return []
    if n == 1:
        return [0]
    fib = [0, 1]
    for i in range(2, n):
        fib.append(fib[-1] + fib[-2])
    return fib""",

    # String functions
    """def is_palindrome(s):
    s = s.lower().replace(' ', '')
    return s == s[::-1]""",

    """def count_vowels(s):
    return sum(1 for c in s.lower() if c in 'aeiou')""",

    """def reverse_words(sentence):
    words = sentence.split()
    return ' '.join(reversed(words))""",

    """def caesar_cipher(text, shift):
    result = ''
    for char in text:
        if char.isalpha():
            base = ord('A') if char.isupper() else ord('a')
            result += chr((ord(char) - base + shift) % 26 + base)
        else:
            result += char
    return result""",

    # List / data structure functions
    """def flatten(nested):
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(flatten(item))
        else:
            result.append(item)
    return result""",

    """def remove_duplicates(lst):
    seen = set()
    result = []
    for item in lst:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result""",

    """def two_sum(nums, target):
    seen = {}
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []""",

    """def stack_push_pop():
    stack = []
    stack.append(1)
    stack.append(2)
    stack.append(3)
    top = stack.pop()
    return top""",

    # File / utility
    """def read_lines(filepath):
    with open(filepath, 'r') as f:
        return [line.strip() for line in f.readlines()]""",

    """def merge_dicts(d1, d2):
    result = d1.copy()
    result.update(d2)
    return result""",

    """def safe_divide(a, b):
    if b == 0:
        raise ValueError('Cannot divide by zero')
    return a / b""",

    """def chunk_list(lst, size):
    return [lst[i:i + size] for i in range(0, len(lst), size)]""",

    """def matrix_multiply(A, B):
    rows_A, cols_A = len(A), len(A[0])
    cols_B = len(B[0])
    C = [[0] * cols_B for _ in range(rows_A)]
    for i in range(rows_A):
        for j in range(cols_B):
            for k in range(cols_A):
                C[i][j] += A[i][k] * B[k][j]
    return C""",

    """def count_words(text):
    words = text.lower().split()
    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1
    return counts""",
]

# ─────────────────────────────────────────────
# Bug injection functions
# ─────────────────────────────────────────────
def inject_off_by_one(code):
    """Change range(n) to range(n+1) or range(n-1)"""
    # Replace first range(something) with an off-by-one version
    def replace_range(match):
        inner = match.group(1)
        return f"range({inner}+1)"
    result = re.sub(r'range\(([^)]+)\)', replace_range, code, count=1)
    return result if result != code else None

def inject_wrong_operator(code):
    """Flip a comparison operator"""
    replacements = [('== 0', '!= 0'), ('> 0', '< 0'), (' >= ', ' > '),
                    (' <= ', ' < '), (' == ', ' != ')]
    for old, new in replacements:
        if old in code:
            return code.replace(old, new, 1)
    return None

def inject_missing_return(code):
    """Remove the return statement from a function"""
    lines = code.split('\n')
    for i, line in enumerate(lines):
        if line.strip().startswith('return ') and i > 0:
            lines[i] = line.replace('return ', '# removed: return ', 1)
            return '\n'.join(lines)
    return None

def inject_wrong_index(code):
    """Replace [0] with [1] or lst[1:] with lst[2:]"""
    if '[0]' in code:
        return code.replace('[0]', '[1]', 1)
    if 'lst[1:]' in code:
        return code.replace('lst[1:]', 'lst[2:]', 1)
    return None

def inject_wrong_variable(code):
    """Swap two similar variable names"""
    pairs = [('left', 'right'), ('a', 'b'), ('i', 'j'), ('max_val', 'item')]
    for v1, v2 in pairs:
        if v1 in code and v2 in code:
            # Swap just the first occurrence of v1
            return code.replace(v1, '___TEMP___', 1).replace(v2, v1, 1).replace('___TEMP___', v2, 1)
    return None

def inject_wrong_arithmetic(code):
    """Change + to - or * to /"""
    if ' + ' in code:
        return code.replace(' + ', ' - ', 1)
    if ' * ' in code:
        return code.replace(' * ', ' / ', 1)
    return None

# All bug injectors
BUG_INJECTORS = [
    inject_off_by_one,
    inject_wrong_operator,
    inject_missing_return,
    inject_wrong_index,
    inject_wrong_variable,
    inject_wrong_arithmetic,
]

def create_buggy_version(code):
    """Try each bug injector until one succeeds"""
    injectors = BUG_INJECTORS.copy()
    random.shuffle(injectors)
    for injector in injectors:
        buggy = injector(code)
        if buggy is not None and buggy != code:
            return buggy
    # Fallback: add a meaningless wrong assignment
    lines = code.split('\n')
    lines.insert(1, '    x = None  # erroneous line')
    return '\n'.join(lines)

# ─────────────────────────────────────────────
# Build the dataset
# ─────────────────────────────────────────────
def build_dataset(clean_examples, multiplier=50):
    """
    For each clean example, generate `multiplier` variations by
    pairing clean code (label=0) with its buggy version (label=1).
    """
    codes, labels = [], []

    for code in clean_examples:
        for _ in range(multiplier):
            # Add clean version
            codes.append(code)
            labels.append(0)  # 0 = clean

            # Add buggy version
            buggy = create_buggy_version(code)
            codes.append(buggy)
            labels.append(1)  # 1 = buggy

    return codes, labels

print("Generating dataset...")
codes, labels = build_dataset(CLEAN_EXAMPLES, multiplier=50)

print(f"Total samples: {len(codes)}")
print(f"Clean samples: {labels.count(0)}")
print(f"Buggy samples: {labels.count(1)}")
print()
print("--- Example clean code ---")
print(codes[0])
print()
print("--- Example buggy version ---")
print(codes[1])

---
## Phase 2 — Preprocessing & Train/Val/Test Split

In [ ]:
# ─────────────────────────────────────────────
# Train / Validation / Test split
# ─────────────────────────────────────────────
# First split off 20% for test
X_temp, X_test, y_temp, y_test = train_test_split(
    codes, labels, test_size=0.10, random_state=SEED, stratify=labels
)

# Then split the remaining into 80% train and 10% val (of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, random_state=SEED, stratify=y_temp
)

print(f"Train size : {len(X_train)}")
print(f"Val size   : {len(X_val)}")
print(f"Test size  : {len(X_test)}")


# ─────────────────────────────────────────────
# Simple tokenizer for the LSTM model
# (CodeBERT has its own tokenizer — see Phase 3B)
# ─────────────────────────────────────────────
def tokenize_code(code_str):
    """
    Use Python's built-in tokenizer to split code into tokens.
    Returns a list of token strings.
    """
    tokens = []
    try:
        token_gen = tokenize.generate_tokens(io.StringIO(code_str).readline)
        for tok_type, tok_string, _, _, _ in token_gen:
            # Skip comments and newlines (they're not semantic)
            if tok_type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE,
                                tokenize.INDENT, tokenize.DEDENT, tokenize.ENDMARKER):
                tokens.append(tok_string)
    except tokenize.TokenError:
        # If tokenization fails, fall back to simple split
        tokens = code_str.split()
    return tokens


def build_vocab(code_list, max_vocab=5000):
    """
    Build a vocabulary from all token strings.
    Returns a dict mapping token -> integer index.
    """
    all_tokens = []
    for code in code_list:
        all_tokens.extend(tokenize_code(code))

    # Count frequencies and keep the top max_vocab tokens
    counter = Counter(all_tokens)
    most_common = counter.most_common(max_vocab - 2)  # reserve 2 slots

    vocab = {'<PAD>': 0, '<UNK>': 1}  # padding and unknown tokens
    for token, _ in most_common:
        vocab[token] = len(vocab)

    return vocab


def encode_code(code_str, vocab, max_len=256):
    """
    Convert code string → fixed-length list of integer IDs.
    Truncates if longer than max_len, pads if shorter.
    """
    tokens = tokenize_code(code_str)
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens]

    # Truncate or pad
    if len(ids) > max_len:
        ids = ids[:max_len]
    else:
        ids = ids + [vocab['<PAD>']] * (max_len - len(ids))

    return ids


# Build vocabulary from training data only (avoid data leakage)
print("Building vocabulary...")
vocab = build_vocab(X_train, max_vocab=5000)
print(f"Vocabulary size: {len(vocab)} unique tokens")

# Encode all splits
MAX_LEN = 256

X_train_enc = [encode_code(c, vocab, MAX_LEN) for c in X_train]
X_val_enc   = [encode_code(c, vocab, MAX_LEN) for c in X_val]
X_test_enc  = [encode_code(c, vocab, MAX_LEN) for c in X_test]

print(f"Encoding done. Each sample is a list of {MAX_LEN} integers.")
print(f"Example first 10 token IDs: {X_train_enc[0][:10]}")

---
## Phase 3A — BiLSTM Model (Baseline)

A BiLSTM reads code tokens in both directions and learns which patterns are suspicious.

In [ ]:
# ─────────────────────────────────────────────
# PyTorch Dataset wrapper
# ─────────────────────────────────────────────
class CodeDataset(Dataset):
    def __init__(self, encoded_codes, labels):
        self.x = torch.tensor(encoded_codes, dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# ─────────────────────────────────────────────
# BiLSTM Architecture
# ─────────────────────────────────────────────
class BiLSTMBugDetector(nn.Module):
    """
    Bidirectional LSTM classifier for bug detection.

    Architecture:
      Embedding → BiLSTM (2 layers) → Dropout → Dense → Softmax
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, num_classes=2):
        super(BiLSTMBugDetector, self).__init__()

        # Embedding layer: maps each token ID to a dense vector
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0  # index 0 is <PAD>, won't contribute to gradients
        )

        # BiLSTM: bidirectional=True means it reads tokens left→right AND right→left
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,         # input shape: (batch, seq_len, features)
            bidirectional=True,       # doubles the output size
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        # Final classifier
        # BiLSTM output size = hidden_dim * 2 (forward + backward)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len)

        embedded = self.embedding(x)          # (batch, seq_len, embed_dim)
        lstm_out, (hidden, _) = self.lstm(embedded)

        # Take the last hidden state from both directions
        # hidden shape: (num_layers * 2, batch, hidden_dim)
        forward_last  = hidden[-2, :, :]  # last layer, forward direction
        backward_last = hidden[-1, :, :]  # last layer, backward direction
        combined = torch.cat([forward_last, backward_last], dim=1)  # (batch, hidden*2)

        dropped = self.dropout(combined)
        out = self.fc(dropped)   # (batch, 2)  → logits for [clean, buggy]
        return out


# ─────────────────────────────────────────────
# Training helper functions
# ─────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += len(batch_y)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += len(batch_y)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels


# ─────────────────────────────────────────────
# Train the BiLSTM
# ─────────────────────────────────────────────
BATCH_SIZE  = 32
LSTM_EPOCHS = 10
LEARNING_RATE = 1e-3

train_ds = CodeDataset(X_train_enc, y_train)
val_ds   = CodeDataset(X_val_enc,   y_val)
test_ds  = CodeDataset(X_test_enc,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

lstm_model = BiLSTMBugDetector(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

print(f"Model parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")
print(f"Training for {LSTM_EPOCHS} epochs...\n")

lstm_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

for epoch in range(1, LSTM_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(lstm_model, train_loader, optimizer, criterion)
    va_loss, va_acc, _, _ = evaluate(lstm_model, val_loader, criterion)

    scheduler.step(va_loss)

    lstm_history['train_loss'].append(tr_loss)
    lstm_history['val_loss'].append(va_loss)
    lstm_history['train_acc'].append(tr_acc)
    lstm_history['val_acc'].append(va_acc)

    # Save the best model checkpoint
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(lstm_model.state_dict(), 'best_lstm_model.pt')

    print(f"Epoch {epoch:2d}/{LSTM_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}")

print(f"\n✅ Best validation accuracy: {best_val_acc:.4f}")

## Visualise BiLSTM Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, LSTM_EPOCHS + 1)

axes[0].plot(epochs_range, lstm_history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(epochs_range, lstm_history['val_loss'],   label='Val Loss',   marker='s')
axes[0].set_title('BiLSTM — Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, lstm_history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(epochs_range, lstm_history['val_acc'],   label='Val Acc',   marker='s')
axes[1].set_title('BiLSTM — Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: lstm_training_curves.png")

## Evaluate BiLSTM on Test Set

In [ ]:
# Load the best checkpoint
lstm_model.load_state_dict(torch.load('best_lstm_model.pt'))

_, test_acc, test_preds, test_labels = evaluate(lstm_model, test_loader, criterion)

print("=" * 50)
print("BiLSTM — Test Set Results")
print("=" * 50)
print(f"Accuracy : {accuracy_score(test_labels, test_preds):.4f}")
print(f"F1 Score : {f1_score(test_labels, test_preds):.4f}")
print(f"Precision: {precision_score(test_labels, test_preds):.4f}")
print(f"Recall   : {recall_score(test_labels, test_preds):.4f}")
print()
print(classification_report(test_labels, test_preds, target_names=['Clean', 'Buggy']))

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Clean', 'Buggy'],
            yticklabels=['Clean', 'Buggy'])
plt.title('BiLSTM — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('lstm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 3B — CodeBERT Fine-tuning (Advanced)

**CodeBERT** is a pretrained transformer model that was trained on millions of lines of
Python, Java, JavaScript, and other languages. Fine-tuning it on our bug detection task
gives much higher accuracy because it already understands code semantics.

- Model: `microsoft/codebert-base` (~125M parameters)
- Training time: ~10–15 minutes on a T4 GPU for 3 epochs
- Expected accuracy: 85–95%

In [ ]:
# ─────────────────────────────────────────────
# Load the CodeBERT tokenizer
# This downloads ~500MB on first run (cached after that)
# ─────────────────────────────────────────────
print("Loading CodeBERT tokenizer... (downloads ~500MB on first run)")
codebert_tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
print("Tokenizer loaded!")


# ─────────────────────────────────────────────
# Custom Dataset for CodeBERT
# ─────────────────────────────────────────────
class CodeBERTDataset(Dataset):
    def __init__(self, codes, labels, tokenizer, max_len=256):
        self.codes    = codes
        self.labels   = labels
        self.tokenizer = tokenizer
        self.max_len  = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.codes[idx],
            max_length=self.max_len,
            padding='max_length',    # pad to max_len
            truncation=True,         # truncate if longer
            return_tensors='pt'      # return PyTorch tensors
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


CODEBERT_MAX_LEN  = 256
CODEBERT_BATCH    = 16   # smaller batch because model is large
CODEBERT_EPOCHS   = 3
CODEBERT_LR       = 2e-5  # standard fine-tuning LR for BERT models

cb_train_ds = CodeBERTDataset(X_train, y_train, codebert_tokenizer, CODEBERT_MAX_LEN)
cb_val_ds   = CodeBERTDataset(X_val,   y_val,   codebert_tokenizer, CODEBERT_MAX_LEN)
cb_test_ds  = CodeBERTDataset(X_test,  y_test,  codebert_tokenizer, CODEBERT_MAX_LEN)

cb_train_loader = DataLoader(cb_train_ds, batch_size=CODEBERT_BATCH, shuffle=True)
cb_val_loader   = DataLoader(cb_val_ds,   batch_size=CODEBERT_BATCH)
cb_test_loader  = DataLoader(cb_test_ds,  batch_size=CODEBERT_BATCH)

print(f"CodeBERT DataLoaders ready.")
print(f"Train batches: {len(cb_train_loader)} | Val batches: {len(cb_val_loader)}")

In [ ]:
# ─────────────────────────────────────────────
# Load CodeBERT model for sequence classification
# RobertaForSequenceClassification adds a classifier head on top of CodeBERT
# ─────────────────────────────────────────────
print("Loading CodeBERT model... (downloads ~500MB on first run)")
codebert_model = RobertaForSequenceClassification.from_pretrained(
    'microsoft/codebert-base',
    num_labels=2  # binary: clean or buggy
).to(device)
print("Model loaded!")
print(f"Total parameters: {sum(p.numel() for p in codebert_model.parameters()):,}")

# Use AdamW optimizer (best practice for transformer fine-tuning)
cb_optimizer = optim.AdamW(codebert_model.parameters(), lr=CODEBERT_LR, weight_decay=0.01)

# Linear learning rate warmup then decay
total_steps = len(cb_train_loader) * CODEBERT_EPOCHS
warmup_steps = total_steps // 10  # warm up for first 10% of steps
cb_scheduler = get_linear_schedule_with_warmup(
    cb_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

cb_criterion = nn.CrossEntropyLoss()


# ─────────────────────────────────────────────
# Training loop for CodeBERT
# ─────────────────────────────────────────────
def train_codebert_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for i, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()

        # CodeBERT returns a SequenceClassifierOutput object
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds  = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total  += len(labels)

        # Print progress every 50 batches
        if (i + 1) % 50 == 0:
            print(f"  Step {i+1}/{len(loader)} — loss: {loss.item():.4f}")

    return total_loss / len(loader), correct / total


def eval_codebert(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = outputs.loss
            logits  = outputs.logits

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += len(labels)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels


print(f"Starting CodeBERT fine-tuning for {CODEBERT_EPOCHS} epochs...")
print(f"(This takes ~10-15 mins on T4 GPU)\n")

cb_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_cb_val_acc = 0

for epoch in range(1, CODEBERT_EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{CODEBERT_EPOCHS} ---")
    tr_loss, tr_acc = train_codebert_epoch(codebert_model, cb_train_loader, cb_optimizer, cb_scheduler)
    va_loss, va_acc, _, _ = eval_codebert(codebert_model, cb_val_loader)

    cb_history['train_loss'].append(tr_loss)
    cb_history['val_loss'].append(va_loss)
    cb_history['train_acc'].append(tr_acc)
    cb_history['val_acc'].append(va_acc)

    if va_acc > best_cb_val_acc:
        best_cb_val_acc = va_acc
        codebert_model.save_pretrained('best_codebert_model')
        codebert_tokenizer.save_pretrained('best_codebert_model')

    print(f"Epoch {epoch} Summary | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}")

print(f"\n✅ Best CodeBERT validation accuracy: {best_cb_val_acc:.4f}")

## Evaluate CodeBERT on Test Set

In [ ]:
# Load best checkpoint
codebert_model = RobertaForSequenceClassification.from_pretrained('best_codebert_model').to(device)

_, cb_test_acc, cb_test_preds, cb_test_labels = eval_codebert(codebert_model, cb_test_loader)

print("=" * 50)
print("CodeBERT — Test Set Results")
print("=" * 50)
print(f"Accuracy : {accuracy_score(cb_test_labels, cb_test_preds):.4f}")
print(f"F1 Score : {f1_score(cb_test_labels, cb_test_preds):.4f}")
print(f"Precision: {precision_score(cb_test_labels, cb_test_preds):.4f}")
print(f"Recall   : {recall_score(cb_test_labels, cb_test_preds):.4f}")
print()
print(classification_report(cb_test_labels, cb_test_preds, target_names=['Clean', 'Buggy']))

# Confusion Matrix
cm = confusion_matrix(cb_test_labels, cb_test_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Clean', 'Buggy'],
            yticklabels=['Clean', 'Buggy'])
plt.title('CodeBERT — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('codebert_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Model Comparison

In [ ]:
models     = ['BiLSTM', 'CodeBERT']
accuracies = [
    accuracy_score(test_labels, test_preds),
    accuracy_score(cb_test_labels, cb_test_preds)
]
f1_scores  = [
    f1_score(test_labels, test_preds),
    f1_score(cb_test_labels, cb_test_preds)
]

x     = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, f1_scores,  width, label='F1 Score',  color='coral')

ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison — Bug Detection')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=2)
ax.bar_label(bars2, fmt='%.3f', padding=2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"BiLSTM  — Accuracy: {accuracies[0]:.4f} | F1: {f1_scores[0]:.4f}")
print(f"CodeBERT — Accuracy: {accuracies[1]:.4f} | F1: {f1_scores[1]:.4f}")

---
## Phase 5 — Interactive Gradio Demo

This cell launches an interactive web UI inside Colab.
Paste any Python function and the model will score it for bugs.

Share the public URL (printed after launch) with your lecturer or classmates!

In [ ]:
import gradio as gr
import torch.nn.functional as F

# ─────────────────────────────────────────────
# Prediction functions
# ─────────────────────────────────────────────
def predict_codebert(code_str):
    """Run CodeBERT inference on a code string."""
    codebert_model.eval()
    encoding = codebert_tokenizer(
        code_str,
        max_length=256,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = codebert_model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = F.softmax(outputs.logits, dim=1)[0]

    clean_prob = probs[0].item()
    buggy_prob = probs[1].item()

    return clean_prob, buggy_prob


def predict_lstm(code_str):
    """Run BiLSTM inference on a code string."""
    lstm_model.eval()
    encoded = encode_code(code_str, vocab, MAX_LEN)
    x = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = lstm_model(x)
        probs  = F.softmax(logits, dim=1)[0]

    return probs[0].item(), probs[1].item()


def analyze_code(code_str, model_choice):
    """Main function called by the Gradio interface."""
    if not code_str.strip():
        return "⚠️ Please enter some Python code.", "", ""

    if model_choice == "CodeBERT (recommended)":
        clean_prob, buggy_prob = predict_codebert(code_str)
        model_name = "CodeBERT"
    else:
        clean_prob, buggy_prob = predict_lstm(code_str)
        model_name = "BiLSTM"

    # Build result string
    verdict = "🐛 BUG DETECTED" if buggy_prob > 0.5 else "✅ LOOKS CLEAN"

    result = f"""{verdict}\n\nModel: {model_name}\nBug probability:   {buggy_prob:.1%}\nClean probability: {clean_prob:.1%}"""

    # Confidence bar (ASCII-style)
    bar_len  = 30
    filled   = int(buggy_prob * bar_len)
    bar      = '█' * filled + '░' * (bar_len - filled)
    bar_str  = f"Bug risk: [{bar}] {buggy_prob:.1%}"

    # Simple tip based on score
    if buggy_prob > 0.8:
        tip = "💡 High confidence: check loop bounds, operators, and return statements."
    elif buggy_prob > 0.5:
        tip = "💡 Moderate risk: review conditional logic and variable usage."
    else:
        tip = "💡 Code looks structurally clean. Double-check edge cases manually."

    return result, bar_str, tip


# ─────────────────────────────────────────────
# Example code snippets for the demo
# ─────────────────────────────────────────────
EXAMPLE_CLEAN = """def find_max(lst):
    if not lst:
        return None
    max_val = lst[0]
    for item in lst[1:]:
        if item > max_val:
            max_val = item
    return max_val"""

EXAMPLE_BUGGY = """def find_max(lst):
    if not lst:
        return None
    max_val = lst[0]
    for item in lst[2:]:       # BUG: should be lst[1:]
        if item < max_val:     # BUG: should be >
            max_val = item
    return max_val"""


# ─────────────────────────────────────────────
# Build the Gradio UI
# ─────────────────────────────────────────────
with gr.Blocks(title="Python Bug Detector") as demo:
    gr.Markdown("# 🐛 Python Code Bug Detector")
    gr.Markdown("Paste any Python function below. The AI will estimate whether it contains a bug.")

    with gr.Row():
        with gr.Column(scale=3):
            code_input = gr.Code(
                label="Python code",
                language="python",
                value=EXAMPLE_CLEAN,
                lines=15
            )
            model_radio = gr.Radio(
                choices=["CodeBERT (recommended)", "BiLSTM (baseline)"],
                value="CodeBERT (recommended)",
                label="Model"
            )
            analyze_btn = gr.Button("🔍 Analyze", variant="primary")

        with gr.Column(scale=2):
            result_box = gr.Textbox(label="Result", lines=6)
            bar_box    = gr.Textbox(label="Risk bar")
            tip_box    = gr.Textbox(label="Tip")

    gr.Examples(
        examples=[
            [EXAMPLE_CLEAN, "CodeBERT (recommended)"],
            [EXAMPLE_BUGGY, "CodeBERT (recommended)"],
        ],
        inputs=[code_input, model_radio]
    )

    analyze_btn.click(
        fn=analyze_code,
        inputs=[code_input, model_radio],
        outputs=[result_box, bar_box, tip_box]
    )

# share=True gives a public URL valid for 72 hours
demo.launch(share=True)

---
## Optional: Use the Real CodeSearchNet Dataset

Once your notebook is working with the synthetic data, replace the data generation cell
with this to train on real Python code from GitHub.

> **Note**: The dataset is ~1GB. It streams automatically so you don't need to download it all.

```python
from datasets import load_dataset

# Load Python split (streaming to avoid downloading 1GB upfront)
ds = load_dataset('code_search_net', 'python', split='train', streaming=True)

# Collect 5000 samples
clean_codes = []
for i, sample in enumerate(ds):
    if i >= 5000:
        break
    clean_codes.append(sample['whole_func_string'])

# Then use build_dataset(clean_codes) to create buggy pairs
codes, labels = build_dataset(clean_codes, multiplier=1)
```

---
## Save Everything to Google Drive

To keep your trained models between sessions:

```python
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('best_codebert_model', '/content/drive/MyDrive/bug_detector/codebert_model')
torch.save(lstm_model.state_dict(), '/content/drive/MyDrive/bug_detector/lstm_model.pt')
print('Saved to Google Drive!')
```